# EDA — Modelo Predictivo de Riesgo Pay

**Objetivo:** Explorar las hojas de la base de datos, identificar variables disponibles, detectar problemas de calidad y sentar las bases para la limpieza de datos.

**Hojas exploradas:**
- `Creditos`
- `Abonos`
- `Clientes`
- `csv_score`

**Referencia:** `data/data-definitions.md`

## 0. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Carga de datos

In [ ]:
FILE_PATH = '../data/Base de datos.xlsx'

df_creditos  = pd.read_excel(FILE_PATH, sheet_name='Creditos')
df_abonos    = pd.read_excel(FILE_PATH, sheet_name='Abonos')
df_clientes  = pd.read_excel(FILE_PATH, sheet_name='Clientes')
df_score     = pd.read_excel(FILE_PATH, sheet_name='csv_score')

# Limpiar espacios extra en nombres de columnas
df_creditos.columns = df_creditos.columns.str.strip()
df_abonos.columns   = df_abonos.columns.str.strip()
df_clientes.columns = df_clientes.columns.str.strip()
df_score.columns    = df_score.columns.str.strip()

print('Creditos: ', df_creditos.shape)
print('Abonos:   ', df_abonos.shape)
print('Clientes: ', df_clientes.shape)
print('csv_score:', df_score.shape)

## 2. Hoja: Creditos

In [ ]:
# Columnas relevantes
cols_creditos = [
    'Portfolio Id', 'Person Id', 'External Code', 'Line', 'Name',
    'Product', 'Is line', 'Opening Date', 'Due Date',
    'Amount', 'Currency', 'Agreed Exchange Rate', 'Loan Status'
]
df_creditos = df_creditos[cols_creditos]
df_creditos.head()

In [ ]:
# Tipos de datos y nulos
df_creditos.info()

In [ ]:
# Valores nulos por columna
df_creditos.isnull().sum()

In [ ]:
# Valores únicos en columnas categóricas clave
print('Products únicos:')
print(df_creditos['Product'].value_counts())
print('\nCurrency:')
print(df_creditos['Currency'].value_counts())
print('\nLoan Status:')
print(df_creditos['Loan Status'].value_counts())
print('\nIs line:')
print(df_creditos['Is line'].value_counts())

In [ ]:
# Filtro: solo productos PAY
df_creditos_pay = df_creditos[df_creditos['Product'].str.contains('PAY', case=False, na=False)]
print(f'Registros totales:    {len(df_creditos)}')
print(f'Registros PAY:        {len(df_creditos_pay)}')

In [ ]:
# Conversión de divisa: Amount a MXN
df_creditos_pay = df_creditos_pay.copy()
df_creditos_pay['Amount_MXN'] = df_creditos_pay.apply(
    lambda r: r['Amount'] * r['Agreed Exchange Rate'] if r['Currency'] == 'DOLARES' else r['Amount'],
    axis=1
)
df_creditos_pay[['Portfolio Id', 'Currency', 'Amount', 'Agreed Exchange Rate', 'Amount_MXN']].head(10)

In [ ]:
# Distribución de Amount_MXN diferenciando líneas de crédito vs ministraciones
lineas        = df_creditos_pay[df_creditos_pay['Is line'] == True]['Amount_MXN'].dropna()
ministraciones = df_creditos_pay[df_creditos_pay['Is line'] == False]['Amount_MXN'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lineas, bins=30, color='steelblue')
axes[0].set_title('Líneas de crédito')
axes[0].set_xlabel('Amount MXN')

axes[1].hist(ministraciones, bins=30, color='darkorange')
axes[1].set_title('Ministraciones')
axes[1].set_xlabel('Amount MXN')

fig.suptitle('Distribución de montos (MXN) — Creditos PAY')
plt.tight_layout()
plt.show()

print(f'Líneas de crédito:  {len(lineas)} registros')
print(f'Ministraciones:     {len(ministraciones)} registros')

## 3. Hoja: Abonos

In [ ]:
# Columnas relevantes
cols_abonos = [
    'Portfolio Id', 'Person Id', 'Full Name', 'Product',
    'Application Date', 'Amount', 'Capital', 'Interest',
    'Penalty', 'Payment Currency', 'Agreed Exchange Rate'
]
df_abonos = df_abonos[cols_abonos]
df_abonos.head()

In [ ]:
df_abonos.info()

In [ ]:
df_abonos.isnull().sum()

In [ ]:
# Filtro: solo productos PAY
df_abonos_pay = df_abonos[df_abonos['Product'].str.contains('PAY', case=False, na=False)]
print(f'Registros totales:    {len(df_abonos)}')
print(f'Registros PAY:        {len(df_abonos_pay)}')

In [ ]:
# Conversión de divisa: Capital a MXN
df_abonos_pay = df_abonos_pay.copy()
df_abonos_pay['Capital_MXN'] = df_abonos_pay.apply(
    lambda r: r['Capital'] * r['Agreed Exchange Rate'] if r['Payment Currency'] == 'DOLARES' else r['Capital'],
    axis=1
)
df_abonos_pay[['Portfolio Id', 'Payment Currency', 'Capital', 'Agreed Exchange Rate', 'Capital_MXN']].head(10)

In [ ]:
# Análisis de Penalty (indicador de morosidad)
print('Registros con Penalty > 0 (se atrasaron):', (df_abonos_pay['Penalty'] > 0).sum())
print('Registros con Penalty = 0 (pagaron a tiempo):', (df_abonos_pay['Penalty'] == 0).sum())
print('\nDistribución de Penalty:')
df_abonos_pay['Penalty'].describe()

## 4. Hoja: Clientes

In [ ]:
# Columnas relevantes
cols_clientes = ['Person Id', 'Full Name', 'Person Type', 'Taxpayer ID Number', 'Person Classification']
df_clientes = df_clientes[cols_clientes]
df_clientes.head()

In [ ]:
df_clientes.info()

In [ ]:
print('Person Type:')
print(df_clientes['Person Type'].value_counts())
print('\nPerson Classification:')
print(df_clientes['Person Classification'].value_counts())

In [ ]:
# Filtros: personas morales y clasificación cliente
# NOTA: ajustar los valores exactos según lo que aparezca en las celdas anteriores
df_clientes_f = df_clientes[
    (df_clientes['Person Type'].str.contains('moral', case=False, na=False)) &
    (df_clientes['Person Classification'].str.contains('cliente', case=False, na=False))
]
print(f'Registros totales:  {len(df_clientes)}')
print(f'Registros filtrados: {len(df_clientes_f)}')

## 5. Hoja: csv_score

In [ ]:
# Columnas relevantes
cols_score = ['FECHA CARGA', 'PRIMERNOMBRE', 'VALORSCORE']
df_score = df_score[cols_score]
df_score.head()

In [ ]:
df_score.info()

In [ ]:
df_score.isnull().sum()

In [ ]:
# Distribución del score PyME
df_score['VALORSCORE'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_score['VALORSCORE'].dropna().plot(kind='hist', bins=20, ax=ax)
ax.set_title('Distribución del Score PyME (Buró de Crédito)')
ax.set_xlabel('VALORSCORE')
plt.tight_layout()
plt.show()

### 5.1 Prueba de fuzzy matching con Clientes

Los nombres en `csv_score` no coinciden exactamente con los de `Clientes`. Se evalúa la coincidencia aproximada.

In [ ]:
# Requiere: pip install rapidfuzz
from rapidfuzz import process, fuzz
import unicodedata, re

def normalizar(nombre):
    if not isinstance(nombre, str):
        return ''
    nombre = nombre.upper().strip()
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    # Quitar palabras genéricas de razones sociales
    for palabra in ['S.A. DE C.V.', 'S DE RL DE CV', 'S.A.P.I.', 'S.A.', 'DE C.V.', 'S.C.']:
        nombre = nombre.replace(palabra, '')
    nombre = re.sub(r'[^A-Z0-9 ]', '', nombre)
    return nombre.strip()

nombres_clientes = df_clientes_f['Full Name'].dropna().tolist()
nombres_clientes_norm = [normalizar(n) for n in nombres_clientes]

def buscar_match(nombre_score, threshold=80):
    nombre_norm = normalizar(nombre_score)
    resultado = process.extractOne(nombre_norm, nombres_clientes_norm, scorer=fuzz.token_sort_ratio)
    if resultado and resultado[1] >= threshold:
        idx = nombres_clientes_norm.index(resultado[0])
        return nombres_clientes[idx], resultado[1]
    return None, None

# Muestra de resultados
muestra = df_score['PRIMERNOMBRE'].dropna().head(20)
resultados = [(n, *buscar_match(n)) for n in muestra]
df_match = pd.DataFrame(resultados, columns=['Nombre Score', 'Match Clientes', 'Similitud'])
df_match

## 6. Resumen de hallazgos

> Completar después de correr el notebook con los datos reales.